In [14]:
import pandas as pd
import feature_engineering_helper as hf


In [15]:
df = pd.read_parquet("../0_data/processed_data/data_with_selected_features_LGB_L.parquet")
df.head()

,SMILES,MP,Type,MP_label,RDKit_TPSA,RDKit_FpDensityMorgan3,RDKit_BCUT2D_MRHI,RDKit_SMR_VSA10,RDKit_VSA_EState4,RDKit_NumRotatableBonds,...,MACCS_142,RDKit_fr_imidazole,RDKit_PEOE_VSA2,RDKit_Chi2n,RDKit_PEOE_VSA1,RDKit_PEOE_VSA11,RDKit_fr_Ar_NH,RDKit_NumHeteroatoms,RDKit_EState_VSA5,RDKit_PEOE_VSA12
0,COc1ccc(cc1)C1(C)CCc2c(-c3c1cc(o3)C)c1c(o2)ccc...,170.00,Train,L,35.51,2.500000,5.958729,10.969244,5.626608,2,...,0,0,0.000000,8.060310,13.571165,0.000000,0,3,16.690354,0.0
1,C[C@H]1[C@@H]2CC[C@@H]3[C@](C1=O)(C2)C(=O)OC[C...,296.85,Train,H,69.67,2.640000,6.074542,17.721856,-1.605879,0,...,0,0,14.383612,9.576021,9.473726,5.783245,0,5,0.000000,0.0
2,Cc1cc(Br)c(cc1Br)C,73.00,Train,L,0.00,1.500000,9.105769,31.859888,2.523102,0,...,0,0,0.000000,2.350576,0.000000,0.000000,0,2,20.072342,0.0
3,OC(=O)c1ccc(c(c1)F)C,170.00,Train,L,37.30,2.636364,5.870666,5.969305,0.433426,1,...,0,0,0.000000,2.269116,5.106527,0.000000,0,3,12.132734,0.0
4,OC(=O)C1CC(=O)c2c1cccc2,116.00,Train,L,54.37,2.769231,6.050513,11.752550,1.217500,1,...,0,0,9.589074,3.105374,5.106527,5.783245,0,3,0.000000,0.0


In [31]:
df_L = df[(df["MP_label"] == "L")]
df_L

,SMILES,MP,Type,MP_label,RDKit_TPSA,RDKit_FpDensityMorgan3,RDKit_BCUT2D_MRHI,RDKit_SMR_VSA10,RDKit_VSA_EState4,RDKit_NumRotatableBonds,...,MACCS_142,RDKit_fr_imidazole,RDKit_PEOE_VSA2,RDKit_Chi2n,RDKit_PEOE_VSA1,RDKit_PEOE_VSA11,RDKit_fr_Ar_NH,RDKit_NumHeteroatoms,RDKit_EState_VSA5,RDKit_PEOE_VSA12
0,COc1ccc(cc1)C1(C)CCc2c(-c3c1cc(o3)C)c1c(o2)ccc...,170.0,Train,L,35.51,2.500000,5.958729,10.969244,5.626608,2,...,0,0,0.000000,8.060310,13.571165,0.000000,0,3,16.690354,0.00000
2,Cc1cc(Br)c(cc1Br)C,73.0,Train,L,0.00,1.500000,9.105769,31.859888,2.523102,0,...,0,0,0.000000,2.350576,0.000000,0.000000,0,2,20.072342,0.00000
3,OC(=O)c1ccc(c(c1)F)C,170.0,Train,L,37.30,2.636364,5.870666,5.969305,0.433426,1,...,0,0,0.000000,2.269116,5.106527,0.000000,0,3,12.132734,0.00000
4,OC(=O)C1CC(=O)c2c1cccc2,116.0,Train,L,54.37,2.769231,6.050513,11.752550,1.217500,1,...,0,0,9.589074,3.105374,5.106527,5.783245,0,3,0.000000,0.00000
5,ClC(C(=O)O)Cl,10.0,Train,L,37.30,1.833333,6.522237,29.171185,0.000000,1,...,0,0,0.000000,0.638934,5.106527,0.000000,0,4,0.000000,4.83627
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17215,CN(CCNCc1cc(ccc1O)[N+](=O)[O-])C,158.5,Test,L,78.64,2.764706,5.423746,5.687386,0.532561,6,...,1,0,10.114318,4.050178,15.323226,0.000000,0,6,18.199101,0.00000
17216,C[Si](C#Cc1ccc(cc1)C#C[Si](C)(C)C)(C)C,120.0,Test,L,0.00,1.111111,6.838306,16.147745,8.934172,0,...,0,0,0.000000,6.577350,0.000000,0.000000,0,2,0.000000,0.00000
17217,Brc1ccc(c(c1)C(F)(F)F)[N+](=O)[O-],34.0,Test,L,43.14,2.357143,9.103000,21.617330,-2.196690,1,...,0,0,10.114318,2.416340,0.000000,0.000000,0,7,6.066367,0.00000
17218,OC(=O)C(C(=O)O)Cc1ccccc1,121.0,Test,L,74.60,2.000000,5.926445,11.938611,0.710463,4,...,0,0,9.589074,2.918784,10.213055,5.917906,0,4,0.000000,0.00000


In [34]:
# Filter low-MP data
df_L = df[df["MP_label"] == "L"]

# Split by Type
df_train = df_L[df_L["Type"] == "Train"]
df_test  = df_L[df_L["Type"] == "Test"]

# Target sizes (70/30 split of 878)
n_total = 878
n_train = int(round(n_total * 0.7))  # 615
n_test  = n_total - n_train          # 263

print(f"Sampling {n_train} Train and {n_test} Test")

# Random undersampling
df_train_sampled = df_train.sample(n=n_train, random_state=42)
df_test_sampled  = df_test.sample(n=n_test, random_state=42)

# Combine back
df_L_sampled = pd.concat([df_train_sampled, df_test_sampled]).reset_index(drop=True)

# Check
print(df_L_sampled["Type"].value_counts(normalize=True))
print(df_L_sampled.shape)

Sampling 615 Train and 263 Test
Type
Train    0.700456
Test     0.299544
Name: proportion, dtype: float64
(878, 93)


In [35]:
non_feature_cols = ['SMILES', 'MP', 'Type', 'MP_label']
data_prefix = '../0_data/processed_data/'

In [36]:
data_with_L_scaled = hf.standardize_data(non_feature_cols=non_feature_cols, dataframe=df_L_sampled, scaler_path=data_prefix + 'L_undersampling_scaler.pkl')
data_with_L_scaled.to_parquet(data_prefix + 'L_undersampling_scaled.parquet', compression= "zstd")


Number of feature columns to standardize: 89
Scaler saved to ../0_data/processed_data/L_undersampling_scaler.pkl


In [37]:
df_L_scaled = pd.read_parquet("../0_data/processed_data/L_undersampling_scaled.parquet")
df_L_scaled

,SMILES,MP,Type,MP_label,RDKit_TPSA,RDKit_FpDensityMorgan3,RDKit_BCUT2D_MRHI,RDKit_SMR_VSA10,RDKit_VSA_EState4,RDKit_NumRotatableBonds,...,MACCS_142,RDKit_fr_imidazole,RDKit_PEOE_VSA2,RDKit_Chi2n,RDKit_PEOE_VSA1,RDKit_PEOE_VSA11,RDKit_fr_Ar_NH,RDKit_NumHeteroatoms,RDKit_EState_VSA5,RDKit_PEOE_VSA12
0,CC1(C)COC(=N1)c1ccccc1,22.0,Train,L,-0.827442,0.493332,-0.405473,-0.802775,-0.312804,-0.610276,...,-0.704521,-0.107299,-0.763826,-0.097865,-0.377674,-0.514213,-0.188025,-0.878498,-0.717886,1.262404
1,n1cncnc1,86.0,Train,L,-0.337013,-1.894165,-1.399312,-1.209711,-0.634594,-0.897657,...,1.419403,-0.107299,-0.763826,-1.290011,-1.051916,-0.514213,-0.188025,-0.516867,0.291522,-0.430279
2,OC[C@@H]1OC([C@H]([C@H]1O)O)O,87.0,Train,L,1.141165,-0.542038,-1.168717,-1.209711,-0.634594,-0.610276,...,-0.704521,-0.107299,-0.763826,-0.736371,2.529766,0.651605,-0.188025,0.206394,-0.717886,-0.430279
3,COC(=O)C1(C)c2ccccc2N(C1=C)C(=O)C,60.0,Train,L,-0.109027,0.564248,-0.338120,0.002171,-0.352727,-0.610276,...,-0.704521,-0.107299,1.506109,0.114592,-0.377674,-0.514213,-0.188025,-0.155237,0.288943,1.265112
4,COc1cc(C(=O)O)c(cc1OC)OC,144.0,Train,L,0.418730,-0.603498,-0.428037,-0.797838,-0.622616,0.251867,...,-0.704521,-0.107299,-0.763826,-0.549756,1.697670,1.617061,-0.188025,0.206394,1.061538,-0.430279
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
873,Nc1ccccc1NC(C(=O)Nc1ccccc1)(c1ccccc1)c1ccccc1,160.0,Test,L,0.480751,-1.033720,-0.344536,0.375141,2.156988,0.826630,...,1.419403,-0.107299,-0.012684,1.109671,1.277787,0.512393,-0.188025,-0.155237,-0.717886,-0.430279
874,FC(c1ncc2c(c1)cccc2)(F)F,31.0,Test,L,-1.077251,0.169146,-0.499006,-0.466427,-0.908704,-0.897657,...,-0.704521,-0.107299,0.016995,-0.471902,-1.051916,-0.514213,-0.188025,-0.155237,-0.388355,-0.430279
875,O1CCOc2ccccc2OCCOCCOc2c(OCC1)cccc2,163.0,Test,L,0.142792,-2.343298,-0.786683,-1.209711,-0.634594,-0.897657,...,-0.704521,-0.107299,-0.763826,0.530125,2.993537,3.748336,-0.188025,0.568025,-0.717886,-0.430279
876,O=P(c1ccccc1)(c1ccccc1)C1CCCCC1,167.0,Test,L,-0.957228,-1.095181,0.865964,0.015081,-0.531355,-0.035514,...,-0.704521,-0.107299,-0.763826,0.572116,-0.402130,-0.514213,-0.188025,-0.878498,0.306439,-0.430279
